# Aula 12 · Lab: criando seu primeiro job (resolução comentada)

## Passo a passo
1. **Jobs & Pipelines → Create → Job** e renomeie para `growdev_pipeline_emprestimos`.
2. Primeira tarefa `preparar_ambiente`: Notebook, Workspace, `Projeto/00_preparar_ambiente`, Serverless.
3. **+ Add task → Notebook** para `bronze`, com **Depends on** = `preparar_ambiente`.
4. Repita para `silver` (depende de `bronze`) e `gold` (depende de `silver`).
5. **Job parameters → Edit** → `catalogo` = `growdev`.
6. **Run now** e acompanhe a aba **Runs**.

### Por que `preparar_ambiente` é uma tarefa?
Ele é **idempotente**: se o catálogo, os schemas e os arquivos já existem, não faz nada. Assim o job funciona até em um workspace novo, sem ninguém precisar lembrar de um passo manual.

## O job como código
No job, clique em **⋮ → View as code → YAML**. O resultado é parecido com este:

```yaml
resources:
  jobs:
    growdev_pipeline_emprestimos:
      name: growdev_pipeline_emprestimos
      parameters:
        - name: catalogo
          default: growdev
      tasks:
        - task_key: preparar_ambiente
          notebook_task:
            notebook_path: /Workspace/Users/<seu_email>/databricks_growdev/Projeto/00_preparar_ambiente
        - task_key: bronze
          depends_on:
            - task_key: preparar_ambiente
          notebook_task:
            notebook_path: /Workspace/Users/<seu_email>/databricks_growdev/Projeto/01_bronze
        - task_key: silver
          depends_on:
            - task_key: bronze
          notebook_task:
            notebook_path: /Workspace/Users/<seu_email>/databricks_growdev/Projeto/02_silver
        - task_key: gold
          depends_on:
            - task_key: silver
          notebook_task:
            notebook_path: /Workspace/Users/<seu_email>/databricks_growdev/Projeto/03_gold
```

> Sem bloco de cluster no YAML = compute **Serverless**.

## Rodando de novo: o que acontece?
Rode o job uma segunda vez e abra a saída da tarefa `bronze`. Vai aparecer **0 linhas novas** em cada tabela: a bronze só ingere arquivos que ainda não viu. A silver faz `MERGE` e a gold faz `overwrite`, então **nada é duplicado**. Isso se chama **idempotência**, e é o que permite agendar o job sem medo (próximo agrupamento).

In [0]:
%sql
SELECT 'silver.loans' AS tabela, COUNT(*) AS linhas, COUNT(DISTINCT loan_id) AS chaves FROM growdev.silver.loans
UNION ALL
SELECT 'silver.customers', COUNT(*), COUNT(DISTINCT customer_id) FROM growdev.silver.customers